# ClinKriya GRPO Training with Unsloth + TRL

Fine-tune **Qwen3-1.7B** on the clinkriya clinical decision-making benchmark using:
- **Unsloth** for 2-4x faster LoRA fine-tuning
- **TRL GRPOTrainer** with environment `tool-use` rollouts
- **Mock FHIR** backend — no live server, runs fully offline
- **Named tool calls** matching benchmark evaluation format

**Runtime:** T4 minimum, A100/H100 recommended

## 1. Install Dependencies & Clone Repo

In [1]:
# Install training stack
!pip install -q unsloth trl datasets huggingface_hub

# Clone repo (contains medagentbench_env + benchmark data)
!git clone https://github.com/ananya173147/clinKriya.git ./repo

# Verify GPU
!nvidia-smi | head -20

fatal: destination path './repo' already exists and is not an empty directory.
Sun Mar  8 19:00:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:DB:00.0 Off |                    0 |
| N/A   26C    P0             82W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                  

In [ ]:
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo


Note: you may need to restart the kernel to use updated packages.


In [4]:
!git -C ./repo pull 

remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 11 (delta 7), reused 11 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 36.23 KiB | 254.00 KiB/s, done.
From https://github.com/ananya173147/clinKriya
   b55b45f..9595f8f  main       -> origin/main
Updating b55b45f..9595f8f
Fast-forward
 OpenEnv_2048_BF16.ipynb         | 8091 +++++++++++++++++++++++++++++++++++++++
 clinKriya_GRPO_BF16.ipynb       |  360 ++
 medagentbench_env/server/app.py |   32 +-
 medagentbench_env/train.py      |   61 +-
 medagentbench_env/ui/index.html |   12 +-
 trainer.ipynb                   |  196 +-
 6 files changed, 8657 insertions(+), 95 deletions(-)
 create mode 100644 OpenEnv_2048_BF16.ipynb
 create mode 100644 clinKriya_GRPO_BF16.ipynb


## 2. Load Model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen3-1.7B"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    dtype = torch.bfloat16
)

print(f"Model: {MODEL_NAME}")
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.13/site-packages/triton/runtime/autotuner.py:101: DeprecationWarning: warmup, rep, and use_cuda_graph parameters are deprecated. See https://github.com/triton-lang/triton/pull/4496 for details.
  warnings.warn(("warmup, rep, and use_cuda_graph parameters are deprecated. See "


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/opt/conda/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=9621) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model: Qwen/Qwen3-1.7B
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## 3. Load Environment & Data

In [ ]:
import importlib.util, sys
from pathlib import Path

REPO = Path("./repo")

# Load train.py directly — avoids installing openenv-core
spec = importlib.util.spec_from_file_location(
    "train", REPO / "medagentbench_env" / "train.py"
)
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

MedAgentTrainEnv = train_mod.MedAgentTrainEnv
reward_func      = train_mod.reward_func
build_dataset    = train_mod.build_dataset

DATA_DIR = REPO / "medagentbench_env" / "data"

# Reset ALL module-level globals before overriding paths.
# If this cell is re-run without a kernel restart, stale cached objects
# (_MOCK_FHIR, _SYSTEM_PROMPT, _TASKS) from the wrong path would persist.
train_mod._MOCK_FHIR         = None
train_mod._SYSTEM_PROMPT     = ""
train_mod._TASKS             = []
train_mod._TASK_INDEX        = 0
train_mod._DATA_DIR          = DATA_DIR
train_mod._CACHE_PATH        = DATA_DIR / "fhir_cache.json"
train_mod._SYSTEM_PROMPT_PATH = DATA_DIR / "new_system.txt"

# Also clear the env registry in case this cell is re-run mid-session
MedAgentTrainEnv._registry.clear()

# Pre-load shared resources (now using the correct paths)
train_mod._get_mock_fhir()
tasks = train_mod._get_tasks()
print(f"FHIR cache loaded | {len(tasks)} tasks")
print(f"System prompt preview: {train_mod._get_system_prompt()[:120]}...")

FHIR cache loaded | 90 tasks
System prompt: You are a medical AI agent that completes clinical tasks by interacting with a FHIR EHR. The patient MRN is always provided in the task — do not search for patients.

You have access to these tools:

- fhir_observation_search(patient, code): search lab results for a patient by LOINC or local code
- fhir_vitals_create(resourceType, category, code, effectiveDateTime, status, valueString, subject): record a vital sign observation
- fhir_service_request_create(resourceType, code, authoredOn, status, intent, priority, subject, note): create a referral or lab order
- finish(value): submit your final answer as a list

Rules:
- Only perform an action when its precondition is explicitly met. If the task says "if X then do Y", only do Y when X is true.
- Always call finish to submit your answer. Never respond with an empty list.
- Numbers must be numbers (not strings). Dates must be ISO format strings. Do not include units.

Good finish examples:
- fin

In [7]:
# Sanity check — run one episode end-to-end
env = MedAgentTrainEnv()
instruction = env.reset()
task = env._task
print(f"Task : {task['id']}  ({task.get('_benchmark_type')})")
print(f"Instr: {instruction[:120]}...")

# Simulate a correct BP observation POST
resp = env.fhir_vitals_create(
    resourceType="Observation",
    category=[{"coding": [{"code": "vital-signs"}]}],
    code={"text": "BP"},
    effectiveDateTime="2023-11-13T10:15:00",
    status="final",
    valueString="118/77",
    subject={"reference": f"Patient/{task['eval_MRN']}"},
)
print(f"POST: {resp[:60]}")

result = env.finish([])
print(f"Finish: {result}")

Task : task3_1  (always-action)
Instr: I just measured the blood pressure for patient with MRN of S2380121, and it is "118/77 mmHg". Help me record it.
Context...
POST: POST request accepted and executed successfully. Please call

────────────────────────────────────────────────────────────
EPISODE TRACE  task=task3_1  steps=2  reward=0.425
────────────────────────────────────────────────────────────
  [1] AGENT: POST http://localhost:8080/fhir/Observation
{"resourceType": "Observation", "status": "final", "category": [{"coding": [{"code": "vital-signs"}]}], "code": {"text": "BP"}, "effectiveDateTime": "2023-11-13T10:15:00", "valueString": "118/77", "subject": {"reference": "Patient/S2380121"}}
  [2] ENV  : POST request accepted and executed successfully. Please call finish if you have got answers for all the questions and finished all the requested tasks
  [3] AGENT: FINISH([])
  [4] ENV  : Task completed.
  ANSWER: []
────────────────────────────────────────────────────────────
Finis

## 4. Build Training Dataset

In [8]:
dataset = build_dataset(DATA_DIR)
print(f"Dataset: {len(dataset)} tasks")
print(f"Roles  : {[m['role'] for m in dataset[0]['prompt']]}")
print(f"User   : {dataset[0]['prompt'][1]['content']}")

Dataset: 90 tasks
Roles  : ['system', 'user']
User   : I just measured the blood pressure for patient with MRN of S2380121, and it is "118/77 mmHg". Help me record it.
Context: It's 2023-11-13T10:15:00+00:00 now. The flowsheet ID for blood pressure is BP.


In [ ]:
# Add this in Cell 14 before creating the trainer:
MedAgentTrainEnv._registry.clear()
train_mod._TASK_INDEX = 0

In [ ]:
def reward_func(prompts, completions, environments=None, **kwargs):
    num_completions = len(completions)
    
    if environments is None:
        environments = kwargs.get("environments")

    if environments is not None:
        envs = environments
    else:
        n_prompts = len(MedAgentTrainEnv._registry)
        assert n_prompts > 0, (
            "Registry is empty — environment_factory was not called or "
            "registry was cleared prematurely. Check Unsloth version."
        )
        envs = MedAgentTrainEnv._registry[:n_prompts]
        del MedAgentTrainEnv._registry[:n_prompts]

    n_prompts = len(envs)
    if n_prompts == 0:
        return [0.0] * num_completions
    
    num_generations = num_completions // n_prompts
    rewards = []
    for env in envs:
        rewards.extend([float(env.reward)] * num_generations)
    
    if len(rewards) < num_completions:
        rewards.extend([0.0] * (num_completions - len(rewards)))
    return rewards[:num_completions]

## 5. Train with GRPOTrainer

In [ ]:
import re, json, logging
from trl import GRPOConfig, GRPOTrainer

# Suppress malformed deprecation warning from transformers 5.2.0
logging.getLogger('transformers.modeling_attn_mask_utils').setLevel(logging.ERROR)

OUTPUT_DIR = './medagent_grpo_output'

# ── Standalone reward function (no environment_factory needed) ────────────────
# Unsloth's _calculate_rewards never passes `environments` to reward_func, so
# we use the 2048-style approach: parse tool calls from the completion text,
# replay them through a fresh env, and return the shaped reward.

def _find_task(prompt_messages):
    """Match prompt user message to a task by its instruction prefix."""
    tasks = train_mod._get_tasks()
    user_content = next(
        (m['content'] for m in prompt_messages
         if isinstance(m, dict) and m.get('role') == 'user'), ''
    )
    for task in tasks:
        needle = task['instruction'][:60]
        if needle and needle in user_content:
            return task
    return None


def _replay(completion_text, task):
    """Create a fresh env for `task`, parse & execute tool calls, return reward."""
    tasks = train_mod._get_tasks()
    idx = next((i for i, t in enumerate(tasks) if t['id'] == task['id']), None)
    if idx is None:
        return 0.0

    # Snapshot and restore task index so we don't disturb the global counter
    old_idx = train_mod._TASK_INDEX
    train_mod._TASK_INDEX = idx
    env = MedAgentTrainEnv()
    env.reset()
    train_mod._TASK_INDEX = old_idx

    # --- Parse tool calls: <tool_call>{...}</tool_call> ---
    tool_calls = []
    for m in re.finditer(r'<tool_call>\s*(\{.*?\})\s*</tool_call>',
                         completion_text, re.DOTALL):
        try:
            tool_calls.append(json.loads(m.group(1)))
        except json.JSONDecodeError:
            pass

    # Fallback: bare {"name": ..., "arguments": ...}
    if not tool_calls:
        for m in re.finditer(
            r'\{\s*"name"\s*:\s*"(\w+)"\s*,\s*"arguments"\s*:\s*(\{.*?\})\s*\}',
            completion_text, re.DOTALL):
            try:
                tool_calls.append({'name': m.group(1),
                                   'arguments': json.loads(m.group(2))})
            except json.JSONDecodeError:
                pass

    # Fallback: finish([...]) bare call
    if not tool_calls:
        m = re.search(r'finish\s*\(\s*(\[.*?\])\s*\)', completion_text, re.DOTALL)
        if m:
            try:
                env.finish(json.loads(m.group(1)))
                return env.reward
            except Exception:
                pass

    for tc in tool_calls:
        if env.done:
            break
        name = tc.get('name', '')
        args = tc.get('arguments', {})
        method = getattr(env, name, None)
        if method and not name.startswith('_'):
            try:
                method(**args)
            except Exception:
                pass

    if not env.done:
        env.finish([])
    return env.reward


def grpo_reward_func(prompts, completions, **kwargs):
    """Self-contained reward: parse completions, replay tool calls, return rewards."""
    rewards = []
    for prompt, completion in zip(prompts, completions):
        # completions is a list of message dicts
        if isinstance(completion, list):
            text = '\n'.join(
                m.get('content', '') for m in completion
                if isinstance(m, dict) and m.get('role') == 'assistant'
            )
        else:
            text = str(completion)

        task = _find_task(prompt)
        if task is None:
            rewards.append(0.0)
            continue
        try:
            rewards.append(_replay(text, task))
        except Exception:
            rewards.append(0.0)
    return rewards


# ── Dataset (5-task smoke-test) ───────────────────────────────────────────────
small_dataset = dataset.select(range(5))

# ── GRPOConfig ────────────────────────────────────────────────────────────────
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_completion_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_steps=2,
    logging_steps=1,
    log_completions=True,
    num_completions_to_print=1,
    bf16=True,
    fp16=False,
    save_steps=50,
    save_total_limit=2,
    num_generations=4,
    beta=0.01,
    temperature=0.9,
)

# ── Trainer (no environment_factory — reward function is self-contained) ──────
trainer = GRPOTrainer(
    model=model,
    reward_funcs=grpo_reward_func,
    train_dataset=small_dataset,
    processing_class=tokenizer,
    args=grpo_config,
)

print('Starting GRPO training on 5 tasks...')
trainer.train()


## 6. Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")

# Merge LoRA into base weights (optional — needed for full model push)
# merged = model.merge_and_unload()
# merged.save_pretrained(OUTPUT_DIR + "_merged")
# tokenizer.save_pretrained(OUTPUT_DIR + "_merged")

## 7. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN    = "hf_xxx"          # your HF write token
HF_REPO_ID  = "YOUR_USERNAME/medagent-qwen3-1.7b"

login(token=HF_TOKEN)

# Push LoRA adapter
model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

## 8. Quick Evaluation

Run the trained model on a sample of tasks. The model generates tool calls;
we parse Qwen3's `<tool_call>` format and route to the environment methods.

In [ ]:
import json, re

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (REPO / "medagentbench_env" / "data" / "new_system.txt").read_text().strip()


def parse_tool_call(text: str):
    """Parse Qwen3 <tool_call>{...}</tool_call> format."""
    m = re.search(r'<tool_call>\s*({.*?})\s*</tool_call>', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # Fallback: bare {"name": ..., "arguments": ...}
    m = re.search(r'\{\s*"name"\s*:\s*"(\w+)"\s*,\s*"arguments"\s*:\s*(\{.*?\})\s*\}',
                  text, re.DOTALL)
    if m:
        try:
            return {"name": m.group(1), "arguments": json.loads(m.group(2))}
        except json.JSONDecodeError:
            pass
    return None


def run_episode(env, max_steps=8):
    """Run one episode with the trained model, return reward and trace."""
    instruction = env.reset()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": instruction},
    ]

    for step in range(max_steps):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        reply = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        tc = parse_tool_call(reply)
        if tc is None:
            # No parseable tool call — end episode
            env.finish([])
            break

        tool_name = tc.get("name", "")
        tool_args = tc.get("arguments", {})
        method = getattr(env, tool_name, None)

        if method is None or tool_name.startswith("_"):
            env.finish([])
            break

        try:
            tool_result = method(**tool_args)
        except Exception as e:
            tool_result = f"Tool error: {e}"

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "tool",      "content": tool_result,
                         "tool_call_id": tool_name})

        if env.done:
            break

    return env.reward


# Reset task counter so we evaluate from the start
train_mod._TASK_INDEX = 0

NUM_EVAL = 10
rewards = []

for i in range(NUM_EVAL):
    env = MedAgentTrainEnv()
    r = run_episode(env)
    rewards.append(r)
    print(f"  {env._task['id']}: reward={r:.3f}")

avg = sum(rewards) / len(rewards)
print(f"\nPost-training avg reward ({NUM_EVAL} tasks): {avg:.4f}")
print(f"Baseline (Qwen3-8B OpenRouter):             0.2748")

## 9. Load from HuggingFace (clone)

Re-load the pushed model on any machine — no repo clone needed.

In [ ]:
# from unsloth import FastLanguageModel
# 
# HF_REPO_ID = "YOUR_USERNAME/medagent-qwen3-1.7b"
# 
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=HF_REPO_ID,
#     max_seq_length=4096,
#     load_in_4bit=True,
# )
# FastLanguageModel.for_inference(model)
# print(f"Loaded {HF_REPO_ID} from HuggingFace Hub")